In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

### Read in batch corrected long read RNA-seq counts from oarfish

In [ ]:
long_read = pd.read_csv("nf-core-plantlongrnaseq/output_desiree_unitato/tximeta/all_samples.gene_counts_combat_seq_corrected.tsv", sep='\t', index_col=0)
long_read_transcript = pd.read_csv("nf-core-plantlongrnaseq/output_desiree_unitato/tximeta/all_samples.transcript_counts_combat_seq_corrected.tsv", sep='\t', index_col=0)
sample_names = long_read.columns[1:]  # Exclude the first column which is gene names

# For each sample compute CPMs
cpm_long_read = long_read[sample_names].div(long_read[sample_names].sum(axis=0), axis=1) * 1e6

corr_spearman_unlog = cpm_long_read['DE_H_H_19'].corr(cpm_long_read['DE_H_H_20'], method='spearman')
cpm_long_read_unlog = cpm_long_read
cpm_long_read = np.log2(cpm_long_read + 1)  # Adding 1 to avoid log(0)
corr_spearman_log = cpm_long_read['DE_H_H_19'].corr(cpm_long_read['DE_H_H_20'], method='spearman')
# For long read transcript counts
cpm_long_read_transcript = long_read_transcript[sample_names].div(long_read_transcript[sample_names].sum(axis=0), axis=1) * 1e6
# apply log2
cpm_long_read_transcript = np.log2(cpm_long_read_transcript + 1)  # Adding 1 to avoid log(0)

# Read in short read from salmon

In [ ]:
short_read = pd.read_csv("Desiree_ASE/output/nf_core_rnaseq_short_read_heat_unitato/salmon/salmon.merged.gene_counts.tsv", sep='\t', index_col=0)
short_read_transcript = pd.read_csv("Desiree_ASE/output/nf_core_rnaseq_short_read_heat_unitato/salmon/salmon.merged.transcript_counts.tsv", sep='\t', index_col=0)
# For each sample compute CPMs
# For each sample compute CPMs
cpm_short_read = short_read[sample_names].div(short_read[sample_names].sum(axis=0), axis=1) * 1e6
cpm_short_read_unlog = cpm_short_read
# apply log2
cpm_short_read = np.log2(cpm_short_read + 1)  # Adding 1 to avoid log(0)
# For short read transcript counts
cpm_short_read_transcript = short_read_transcript[sample_names].div(short_read_transcript[sample_names].sum(axis=0), axis=1) * 1e6

# apply log2
cpm_short_read_transcript = np.log2(cpm_short_read_transcript + 1)  # Adding 1 to avoid log(0)

# Merge the two CPM DataFrames
cpm_merged = pd.merge(cpm_long_read, cpm_short_read, how='outer', suffixes=('_long', '_short'), left_index=True, right_index=True)
cpm_merged_transcript = pd.merge(cpm_long_read_transcript, cpm_short_read_transcript, how='outer', suffixes=('_long', '_short'), left_index=True, right_index=True)



# only keep the genes that have in all samples at least 1 cpm
cpm_merged = cpm_merged[(cpm_merged > 1).all(axis=1)]
cpm_merged_transcript = cpm_merged_transcript[(cpm_merged_transcript > 1).all(axis=1)]

## Check raw read counts

In [ ]:
# Method 2: Separate subplots
samples = ['DE_H_H_19', 'DE_H_H_20']  # Add your sample names
n_samples = len(samples)

fig, axes = plt.subplots(1, n_samples, figsize=(7, 3))

for i, sample in enumerate(samples):
    axes[i].hist(cpm_long_read_unlog[sample], bins=100, alpha=0.7)
    axes[i].set_xlabel('Read Counts')
    axes[i].set_ylabel('Frequency')
    axes[i].set_yscale('log')
    axes[i].set_title(sample)
    axes[i].set_xlim(0, 15000)  # Set y-axis limit to start from 1

      

plt.tight_layout()
# Set y-axis limit to start from 1
plt.show()

In [ ]:
# total counts per sample
print(cpm_long_read[sample_names].sum(axis=0))

In [ ]:
print(cpm_short_read[sample_names].sum(axis=0))

In [ ]:
# Method 2: Separate subplots
samples = ['DE_H_H_19', 'DE_H_H_20']  # Add your sample names
n_samples = len(samples)

fig, axes = plt.subplots(1, n_samples, figsize=(7, 3))

for i, sample in enumerate(samples):
    axes[i].hist(cpm_short_read_unlog[sample], bins=100, alpha=0.7)
    axes[i].set_xlabel('Read Counts')
    axes[i].set_ylabel('Frequency')
    axes[i].set_yscale('log')
    axes[i].set_title(sample)
    

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde

# Assuming you have your data in a numpy array or pandas DataFrame
# x should be your count matrix with samples as columns

# Get number of samples
nsamples = cpm_long_read.shape[1]

# Generate colors (Paired colormap equivalent)
colors = sns.color_palette("Paired", nsamples)

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 2.5))

# Plot A: Raw data
for i in range(nsamples):
    kde = gaussian_kde(cpm_long_read.iloc[:, i])
    x_range = np.linspace(cpm_long_read.iloc[:, i].min(), cpm_long_read.iloc[:, i].max(), 200)
    density = kde(x_range)
    ax1.plot(x_range, density, color=colors[i], linewidth=2, 
             label=cpm_long_read.columns[i])

ax1.axvline(x=0, linestyle='--', color='gray', linewidth=1)

ax1.set_xlabel('Log-cpm')
ax1.set_title('A. LONG: Raw data')
ax1.legend(loc='upper right', frameon=False)
ax1.tick_params(axis='y', rotation=0)

#filter data where in at least 3 samples cpm > 1
cpm_long_read_filtered = cpm_long_read[(cpm_long_read > 3).sum(axis=1) >= 7]
# x should be your count matrix with samples as columns

#cpm_long_read_filtered = cpm_long_read_filtered[(cpm_long_read_filtered < 8).sum(axis=1) >= 7]

# Plot B: Filtered data
for i in range(nsamples):
    kde = gaussian_kde(cpm_long_read_filtered.iloc[:, i])
    x_range = np.linspace(cpm_long_read_filtered.iloc[:, i].min(), cpm_long_read_filtered.iloc[:, i].max(), 200)
    density = kde(x_range)
    ax2.plot(x_range, density, color=colors[i], linewidth=2, 
             label=cpm_long_read_filtered.columns[i])  
ax2.axvline(x=0, linestyle='--', color='gray', linewidth=1)

ax2.set_xlabel('Log-cpm')
ax2.set_title('B. LONG: Filtered data')
ax2.legend(loc='upper right', frameon=False)

In [ ]:
# count the number of genes with cpm > 0 for each sample
gene_counts = (cpm_long_read  > 0).sum(axis=0)
print(gene_counts)

In [ ]:
#short_read.drop('gene_name', axis=1, inplace=True)
# gene_counts = (short_read > 0).sum(axis=0)
# print(gene_counts)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 2.5))
# Plot B: Filtered data
for i in range(0,4):
    kde = gaussian_kde(cpm_long_read_filtered.iloc[:, i])
    x_range = np.linspace(cpm_long_read_filtered.iloc[:, i].min(), cpm_long_read_filtered.iloc[:, i].max(), 200)
    density = kde(x_range)
    ax1.plot(x_range, density, color=colors[i], linewidth=2, 
             label=cpm_long_read_filtered.columns[i])  
ax1.axvline(x=0, linestyle='--', color='gray', linewidth=1)

ax1.set_xlabel('Log-cpm')
#ax1.set_ylim(0, 0.22)
ax1.set_yticks(np.arange(0, 0.25, 0.05))
ax1.set_title('Long: control samples')
ax1.legend(loc='upper right', frameon=False)


for i in [4,5,6,7]:
    kde = gaussian_kde(cpm_long_read_filtered.iloc[:, i])
    x_range = np.linspace(cpm_long_read_filtered.iloc[:, i].min(), cpm_long_read_filtered.iloc[:, i].max(), 200)
    density = kde(x_range)
    ax2.plot(x_range, density, color=colors[i], linewidth=2, 
             label=cpm_long_read_filtered.columns[i])  
ax2.axvline(x=0, linestyle='--', color='gray', linewidth=1)

ax2.set_xlabel('Log-cpm')
ax2.set_yticks(np.arange(0, 0.25, 0.05))
ax2.set_title('long: heat samples')
ax2.legend(loc='upper right', frameon=False)



In [ ]:

# Assuming you have your data in a numpy array or pandas DataFrame
# x should be your count matrix with samples as columns
cpm_short_read_filtered = cpm_short_read[(cpm_short_read > 3).sum(axis=1) >= 7]
#cpm_short_read_filtered = cpm_short_read_filtered[(cpm_short_read_filtered < 8).sum(axis=1) >= 7]

# Get number of samples
nsamples = cpm_short_read.shape[1]

# Generate colors (Paired colormap equivalent)
colors = sns.color_palette("Paired", nsamples)

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 2.5))

# Plot A: Raw data
for i in range(0,4):
    kde = gaussian_kde(cpm_short_read_filtered.iloc[:, i])
    x_range = np.linspace(cpm_short_read_filtered.iloc[:, i].min(), cpm_short_read_filtered.iloc[:, i].max(), 200)
    density = kde(x_range)
    ax1.plot(x_range, density, color=colors[i], linewidth=2, 
             label=cpm_short_read_filtered.columns[i])
ax1.axvline(x=0, linestyle='--', color='gray', linewidth=1)
ax1.set_xlabel('Log-cpm')
ax1.set_title('short: control samples')
ax1.legend(loc='upper right', frameon=False)
ax1.tick_params(axis='y', rotation=0)

#filter data where in at least 3 samples cpm > 1


# Plot B: Filtered data
for i in [4,5,6,7]:
    kde = gaussian_kde(cpm_short_read_filtered.iloc[:, i])
    x_range = np.linspace(cpm_short_read_filtered.iloc[:, i].min(), cpm_short_read_filtered.iloc[:, i].max(), 200)
    density = kde(x_range)
    ax2.plot(x_range, density, color=colors[i], linewidth=2, 
             label=cpm_short_read_filtered.columns[i])  
ax2.axvline(x=0, linestyle='--', color='gray', linewidth=1)

ax2.set_xlabel('Log-cpm')
ax2.set_title('short: heat samples')
ax2.legend(loc='upper right', frameon=False)



In [ ]:
import pandas as pd

samplenames = cpm_short_read.columns.tolist()
# Convert to DataFrame for easier plotting
lcpm_df = pd.DataFrame(cpm_long_read_filtered, columns=samplenames)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))

# Create boxplot with colors
bp1 = lcpm_df.boxplot(ax=ax1, column=samplenames, 
                      return_type='dict', patch_artist=True)

# Apply colors to boxes
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color)

ax1.set_ylabel('Log-cpm')
ax1.set_title('long read filtered data')
ax1.tick_params(axis='x', rotation=90)
ax1.set_ylim([-1, 17])

# add for long-read filtered data the boxplot
scpm_df = pd.DataFrame(cpm_short_read_filtered, columns=samplenames)
bp2 = scpm_df.boxplot(ax=ax2, column=samplenames, 
                      return_type='dict', patch_artist=True)

# Apply colors to boxes
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
ax2.set_ylabel('Log-cpm')
ax2.set_title('short read filtered data')
ax2.tick_params(axis='x', rotation=90)
ax2.set_ylim([-1, 17])
plt.tight_layout()
plt.show()



In [ ]:
cpm_long_read['DE_C_H_13'].sort_values(ascending=False).head()


In [ ]:
cpm_short_read['DE_C_H_13'].sort_values(ascending=False).head()

In [ ]:
# get the gene name with the highest count in long read sample DE_H_H_19
long_high_genes = []
for sample in long_read.columns[1:-1]:
    #print(long_read[sample].sort_values(ascending=False).head(20))
    long_high_genes.append(long_read[sample].sort_values(ascending=False).head(20).index.tolist())

# After your existing code that builds long_high_genes list:

# Find genes that appear in ALL samples' top 20
shared_genes = set(long_high_genes[0])
for gene_list in long_high_genes[1:]:
    shared_genes = shared_genes.intersection(set(gene_list))

print(f"Number of shared genes across all samples: {len(shared_genes)}")
print("Shared genes:", sorted(shared_genes))

In [ ]:
# get the gene name with the highest count in long read sample DE_H_H_19
short_high_genes = []
for sample in short_read.columns[1:-1]:
    #print(short_read[sample].sort_values(ascending=False).head(20))
    short_high_genes.append(short_read[sample].sort_values(ascending=False).head(20).index.tolist())

# Find genes that appear in ALL samples' top 20
shared_genes = set(short_high_genes[0])
for gene_list in short_high_genes[1:]:
    shared_genes = shared_genes.intersection(set(gene_list))

print(f"Number of shared genes across all samples: {len(shared_genes)}")
print("Shared genes:", sorted(shared_genes))

In [ ]:
short_read['DE_H_H_20'].sort_values(ascending=False).head(10)

#### PCA of all

In [ ]:
import plotly.express as px
from sklearn.decomposition import PCA


In [ ]:
# only first 16 
# Step 1: Select first 16 columns
cpm_first16 = cpm_merged.iloc[:, :16]

# Step 2: Filter rows where all values in these 16 columns are >= 2
cpm_filtered = cpm_first16[(cpm_first16 >= 2).all(axis=1)]

# Step 3: Transpose and convert to array
X = cpm_filtered.T.values

pca = PCA(n_components=2)
components = pca.fit_transform(X)
# add the proportion of variance explained by each component
explained_variance = pca.explained_variance_ratio_
print(f'Explained variance by component 1: {explained_variance[0]:.2%}')
print(f'Explained variance by component 2: {explained_variance[1]:.2%}')
# make the same color for frist 4 samples
colors = ['control_Kinnex']*4 + ['heat_Kinnex']*4 + ['control_Illumina']*4 + ['heat_Illumina']*4

fig = px.scatter(components, x=0, y=1, color= colors, labels = cpm_merged.T.index[:16], )
# add the proportion of variance explained by each component to the axis labels
fig.update_layout(
    xaxis_title=f'PC1 ({explained_variance[0]:.2%} variance)',
    yaxis_title=f'PC2 ({explained_variance[1]:.2%} variance)'
)

# save as svg
fig.write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/PCA_short_vs_long_read_batch_corrected.svg", width = 6, height = 4)
fig.show()


In [ ]:
cpm_merged[:16]


#### Correlation of gene counts between samples

In [ ]:
plt.figure(figsize=(3, 3))
plt.scatter(cpm_merged['DE_C_H_13_long'], cpm_merged['DE_C_H_14_long'], alpha=0.5) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5)

# add the pearson correlation coefficient to the plot
corr = cpm_merged['DE_C_H_13_long'].corr(cpm_merged['DE_C_H_14_long'])
corr_spearman = cpm_merged['DE_C_H_13_long'].corr(cpm_merged['DE_C_H_14_long'], method='spearman')
# plot the correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))


plt.xlabel('Kinnex Gene DE_C_H_13 (log2CPM)')
plt.ylabel('Kinnex Gene DE_C_H_14 (log2CPM)')

plt.title('Kinnex gene counts comparison')
plt.xlim(-1, 15)
plt.ylim(-1, 15)
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))

In [ ]:
# add the gene lengths to the dataframe
gene_lengths = pd.read_csv("LR_DESIREE_PAPER/DEG_results/avgTxlengths_desiree_unitato.tsv", sep='\t')# rename the columns
gene_lengths['avgTxLength'] = gene_lengths[gene_lengths.columns[0]]
# drop the first column 
gene_lengths.drop(gene_lengths.columns[0], axis=1, inplace=True)
cpm_merged = cpm_merged.join(gene_lengths, how='left')

# add gene length categories 
cpm_merged['length_category'] = pd.cut(cpm_merged['avgTxLength'], bins=[0, 750, 1250, 3500, np.inf], labels=['0-0.75kb', '0.75-1.25kb', '1.25-3.5kb', '>3.5kb'])
# relace NaN with 'unknown'
cpm_merged['length_category'] = cpm_merged['length_category'].cat.add_categories(['unknown']).fillna('unknown')

In [ ]:
cpm_merged.value_counts('length_category')

In [ ]:
plt.figure(figsize=(3, 3))
plt.scatter(cpm_merged['DE_C_H_13_long'], cpm_merged['DE_H_H_17_long'], alpha=0.5) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5)

# add the pearson correlation coefficient to the plot
corr = cpm_merged['DE_C_H_13_long'].corr(cpm_merged['DE_H_H_17_long'])
corr_spearman = cpm_merged['DE_C_H_13_long'].corr(cpm_merged['DE_H_H_17_long'], method='spearman')
# plot the correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))


plt.xlabel('Kinnex Gene DE_C_H_13 (log2CPM)')
plt.ylabel('Kinnex Gene DE_H_H_17 (log2CPM)')

plt.title('Kinnex gene counts comparison')
plt.xlim(-1, 15)
plt.ylim(-1, 15)
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))

In [ ]:
import matplotlib.pyplot as plt
import numpy as pd

# Define sample pairs
control_samples = ["DE_C_H_13", "DE_C_H_14", "DE_C_H_15", "DE_C_H_16"]
treated_samples = ["DE_H_H_17", "DE_H_H_18", "DE_H_H_19", "DE_H_H_20"]

# Get categories
categories = cpm_merged['length_category'].unique()
categories = categories.dropna()

# ============= LONG vs LONG =============
fig, axes = plt.subplots(4, 4, figsize=(10, 10))


for row_idx, (ctrl, treat) in enumerate(zip(control_samples, treated_samples)):
    for col_idx, category in enumerate(categories):
        ax = axes[row_idx, col_idx]
        data = cpm_merged[cpm_merged['length_category'] == category]
        
        ax.scatter(data[f'{ctrl}_long'], data[f'{treat}_long'], alpha=0.005,  )
        
        # Calculate correlation
        corr = data[f'{ctrl}_long'].corr(data[f'{treat}_long'])
        
        # Add text
        ax.text(0.05, 0.95, f'r = {corr:.3f}\nn = {len(data)}', 
                transform=ax.transAxes, fontsize=9, verticalalignment='top', 
                bbox=dict(facecolor='white', alpha=0.5))
        
        # Labels only on edges
        if col_idx == 0:
            ax.set_ylabel(f'{treat}\n(log2CPM)', fontsize=9)
        if row_idx == 3:
            ax.set_xlabel(f'{ctrl}\n(log2CPM)', fontsize=9)
        
        # Title only on top row
        if row_idx == 0:
            ax.set_title(f'{category}', fontsize=10)
        
 
        ax.set_xticks(np.arange(0, 1, 1))
        ax.set_yticks(np.arange(0, 1, 1))
        ax.set_xlim(-1, 16)
        ax.set_ylim(-1, 16)
        
        # Add diagonal line
        ax.plot([-1, 4], [-1, 4], 'r--', alpha=0.3, linewidth=0.5)

plt.suptitle('Long Reads: Control vs Treated by Length Category', fontsize=16, y=0.995)
plt.tight_layout()
plt.show()

# ============= SHORT vs SHORT =============
fig, axes = plt.subplots(4, 4, figsize=(10, 10))


for row_idx, (ctrl, treat) in enumerate(zip(control_samples, treated_samples)):
    for col_idx, category in enumerate(categories):
        ax = axes[row_idx, col_idx]
        data = cpm_merged[cpm_merged['length_category'] == category]
        
        ax.scatter(data[f'{ctrl}_short'], data[f'{treat}_short'], alpha=0.005)
        
        # Calculate correlation
        corr = data[f'{ctrl}_short'].corr(data[f'{treat}_short'])
        
        # Add text
        ax.text(0.05, 0.95, f'r = {corr:.3f}\nn = {len(data)}', 
                transform=ax.transAxes, fontsize=9, verticalalignment='top', 
                bbox=dict(facecolor='white', alpha=0.5))
        
        # Labels only on edges
        if col_idx == 0:
            ax.set_ylabel(f'{treat}\n(log2CPM)', fontsize=9)
        if row_idx == 3:
            ax.set_xlabel(f'{ctrl}\n(log2CPM)', fontsize=9)
        
        # Title only on top row
        if row_idx == 0:
            ax.set_title(f'{category}', fontsize=10)
        
        
        ax.set_xticks(np.arange(0, 1, 1))
        ax.set_yticks(np.arange(0, 1, 1))
        ax.set_xlim(-1, 16)
        ax.set_ylim(-1, 16)
        
        # Add diagonal line
        ax.plot([-1, 4], [-1, 4], 'r--', alpha=0.3, linewidth=0.5)

plt.suptitle('Short Reads: Control vs Treated by Length Category', fontsize=16, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# add category of regulation


import pandas as pd
# read in file 
regulation_df = pd.read_csv("LR_DESIREE_PAPER/DEG_results/limma_short_vs_long_batch_corrected_regulation_pattern.csv")

regulation_df
# merge with cpm_merged
cpm_merged = cpm_merged.merge(regulation_df[['gene_id', 'RegulationCategory']], left_index=True, right_on='gene_id', how='left')
cpm_merged.set_index('gene_id', inplace=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define sample pairs
control_samples = ["DE_C_H_13", "DE_C_H_14", "DE_C_H_15", "DE_C_H_16"]
treated_samples = ["DE_H_H_17", "DE_H_H_18", "DE_H_H_19", "DE_H_H_20"]
# set 300 dpi for the plots
# ============= LONG vs LONG =============
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
axes = axes.flatten()

for idx, (ctrl, treat) in enumerate(zip(control_samples, treated_samples)):
    ax = axes[idx]
    
    data = cpm_merged.dropna(subset=[f'{ctrl}_long', f'{treat}_long', 'RegulationCategory'])
    unique_patterns = data['RegulationCategory'].unique()
    n_patterns = len(unique_patterns)
    colors = plt.cm.Set3(np.linspace(0, 1, n_patterns))
    color_map = {pattern: colors[i] for i, pattern in enumerate(unique_patterns)}

    # Use the color map in your plot
    colors = data['RegulationCategory'].map(color_map)


    ax.scatter(data[f'{ctrl}_long'], data[f'{treat}_long'], alpha=0.3, s=5, color = colors)
    
    # Calculate correlation
    corr = data[f'{ctrl}_long'].corr(data[f'{treat}_long'])
    
    # Add text
    ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}\nn = {len(data)}', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', 
            bbox=dict(facecolor='white', alpha=0.5))
    
    ax.set_xlabel(f'{ctrl} (log2CPM)', fontsize=10)
    ax.set_ylabel(f'{treat} (log2CPM)', fontsize=10)
    ax.set_title(f'{ctrl} vs {treat}', fontsize=11)
    
    
    ax.set_xticks(np.arange(0, 16, 4))
    ax.set_yticks(np.arange(0, 16, 4))
    ax.set_xlim(-1, 16)
    ax.set_ylim(-1, 16)
    # add legend for colors
    from matplotlib.patches import Patch
    handles = [Patch(color=color_map[pattern], label=pattern) 
           for pattern in unique_patterns]
    
    
    
    # Add diagonal line
    ax.plot([-1, 15], [-1, 15], 'r--', alpha=0.3, linewidth=1)
ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.05, 1), fontsize=8)
plt.suptitle('Kinnex: Control vs Heat', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# set 300 dpi for better resolution
plt.rcParams['figure.dpi'] = 150
# Define sample pairs
control_samples = ["DE_C_H_13", "DE_C_H_14", "DE_C_H_15", "DE_C_H_16"]
treated_samples = ["DE_H_H_17", "DE_H_H_18", "DE_H_H_19", "DE_H_H_20"]

# ============= LONG vs LONG =============
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
axes = axes.flatten()

for idx, (ctrl, treat) in enumerate(zip(control_samples, treated_samples)):
    ax = axes[idx]
    
    data = cpm_merged.dropna(subset=[f'{ctrl}_short', f'{treat}_short', 'RegulationCategory'])
    unique_patterns = data['RegulationCategory'].unique()
    n_patterns = len(unique_patterns)
    colors = plt.cm.Set3(np.linspace(0, 1, n_patterns))
    color_map = {pattern: colors[i] for i, pattern in enumerate(unique_patterns)}

    # Use the color map in your plot
    colors = data['RegulationCategory'].map(color_map)


    ax.scatter(data[f'{ctrl}_short'], data[f'{treat}_short'], alpha=0.3, s=5, color = colors)
    
    # Calculate correlation
    corr = data[f'{ctrl}_short'].corr(data[f'{treat}_short'])
    
    # Add text
    ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}\nn = {len(data)}', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', 
            bbox=dict(facecolor='white', alpha=0.5))
    
    ax.set_xlabel(f'{ctrl} (log2CPM)', fontsize=10)
    ax.set_ylabel(f'{treat} (log2CPM)', fontsize=10)
    ax.set_title(f'{ctrl} vs {treat}', fontsize=11)
    
    
    ax.set_xticks(np.arange(0, 16, 4))
    ax.set_yticks(np.arange(0, 16, 4))
    ax.set_xlim(-1, 16)
    ax.set_ylim(-1, 16)
    # add legend for colors
    from matplotlib.patches import Patch
    handles = [Patch(color=color_map[pattern], label=pattern) 
           for pattern in unique_patterns]
    
   
    # Place legend to the right of the plot
    
    # Add diagonal line
    ax.plot([-1, 15], [-1, 15], 'r--', alpha=0.3, linewidth=1)
ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.05, 1), fontsize=8)
plt.suptitle('Illumina: Control vs Heat', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define all samples
samples = ["DE_C_H_13", "DE_C_H_14", "DE_C_H_15", "DE_C_H_16",
           "DE_H_H_17", "DE_H_H_18", "DE_H_H_19", "DE_H_H_20"]

# Get regulation patterns
patterns = cpm_merged['RegulationCategory'].unique()
patterns = patterns[~pd.isna(patterns)]  # Remove NaN if any

# Create one large figure: 8 samples x n patterns
n_patterns = len(patterns)
fig, axes = plt.subplots(8, n_patterns, figsize=(3*n_patterns, 20))

# Get categories for coloring
categories = cpm_merged['length_category'].unique()
categories = categories.dropna()
n_categories = len(categories)
colors_cat = plt.cm.Set3(np.linspace(0, 1, n_categories))
color_map = {category: colors_cat[i] for i, category in enumerate(categories)}

for row_idx, sample in enumerate(samples):
    for col_idx, pattern in enumerate(patterns):
        ax = axes[row_idx, col_idx]
        
        # Filter data by regulation pattern
        data = cpm_merged[cpm_merged['RegulationCategory'] == pattern]
        
        
        # Plot short vs long
        ax.scatter(data[f'{sample}_short'], data[f'{sample}_long'], 
                  alpha=0.05)
        
        # Calculate correlation
        corr = data[f'{sample}_short'].corr(data[f'{sample}_long'])
        
        # Add correlation text
        ax.text(0.05, 0.95, f'r = {corr:.3f}\nn = {len(data)}', 
                transform=ax.transAxes, fontsize=8, verticalalignment='top', 
                bbox=dict(facecolor='white', alpha=0.5))
        
        # Labels only on edges
        if col_idx == 0:
            ax.set_ylabel(f'{sample}\nLong (log2CPM)', fontsize=9)
        if row_idx == 7:
            ax.set_xlabel('Short (log2CPM)', fontsize=9)
        
        # Pattern titles only on top row
        if row_idx == 0:
            ax.set_title(f'{pattern}', fontsize=10)
        
        ax.set_xlim(-1, 15)
        ax.set_ylim(-1, 15)
        ax.set_xticks(np.arange(0, 16, 4))
        ax.set_yticks(np.arange(0, 16, 4))
        
        # Add diagonal reference line
        ax.plot([-1, 15], [-1, 15], 'r--', alpha=0.3, linewidth=0.5)

# Create legend for length categories
handles = [plt.Line2D([0], [0], marker='o', color='w', 
                     markerfacecolor=color_map[cat], markersize=8, label=cat) 
          for cat in categories]
axes[0, -1].legend(handles=handles, loc='upper left', bbox_to_anchor=(1.05, 1), fontsize=8)

plt.suptitle('Short vs Long Read Comparison by Regulation Pattern', fontsize=16, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

categories = cpm_merged['length_category'].unique()
# drop Na values
categories = categories.dropna()

for idx, category in enumerate(categories):
    ax = axes[idx]
    data = cpm_merged[cpm_merged['length_category'] == category]
    
    ax.scatter(data['DE_H_H_17_short'], data['DE_H_H_17_long'], alpha=0.005)
    
    # Calculate correlations
    corr = data['DE_H_H_17_short'].corr(data['DE_H_H_17_long'])
    corr_spearman = data['DE_H_H_17_short'].corr(data['DE_H_H_17_long'], method='spearman')
    
    # Add text
    ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}\nn = {len(data)}', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', 
            bbox=dict(facecolor='white', alpha=0.5))
    
    ax.set_xlabel('DE_H_H_17 (short) (log2CPM)')
    ax.set_ylabel('DE_H_H_17 (long) (log2CPM)')
    ax.set_title(f'{category}')
    ax.set_xlim(-1, 15)
    ax.set_ylim(-1, 15)
    ax.set_xticks(np.arange(0, 16, 4))
    ax.set_yticks(np.arange(0, 16, 4))

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

categories = cpm_merged['length_category'].unique()
# drop Na values
categories = categories.dropna()

for idx, category in enumerate(categories):
    ax = axes[idx]
    data = cpm_merged[cpm_merged['length_category'] == category]
    
    ax.scatter(data['DE_H_H_17_short'], data['DE_C_H_13_long'], alpha=0.005)
    
    # Calculate correlations
    corr = data['DE_C_H_13_short'].corr(data['DE_C_H_13_long'])
    corr_spearman = data['DE_C_H_13_short'].corr(data['DE_C_H_13_long'], method='spearman')
    
    # Add text
    ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}\nn = {len(data)}', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', 
            bbox=dict(facecolor='white', alpha=0.5))
    
    ax.set_xlabel('DE_C_H_13 (short) (log2CPM)')
    ax.set_ylabel('DE_C_H_13 (long) (log2CPM)')
    ax.set_title(f'{category}')
    ax.set_xlim(-1, 15)
    ax.set_ylim(-1, 15)
    ax.set_xticks(np.arange(0, 16, 4))
    ax.set_yticks(np.arange(0, 16, 4))

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))

categories = cpm_merged['length_category'].unique()
# drop Na values
categories = categories.dropna()

for idx, category in enumerate(categories):
    ax = axes[idx]
    data = cpm_merged[cpm_merged['length_category'] == category]
    
    ax.scatter(data['DE_C_H_13_long'], data['DE_H_H_17_long'], alpha=0.005)
    
    # Calculate correlations
    corr = data['DE_C_H_13_long'].corr(data['DE_H_H_17_long'])
    corr_spearman = data['DE_C_H_13_long'].corr(data['DE_H_H_17_long'], method='spearman')
    
    # Add text
    ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}\nn = {len(data)}', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', 
            bbox=dict(facecolor='white', alpha=0.5))
    
    ax.set_xlabel('DE_C_H_13 (long) (log2CPM)')
    ax.set_ylabel('DE_H_H_17 (long) (log2CPM)')
    ax.set_title(f'{category}')
    ax.set_xlim(-1, 15)
    ax.set_ylim(-1, 15)
    ax.set_xticks(np.arange(0, 16, 4))
    ax.set_yticks(np.arange(0, 16, 4))

plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(3, 3))
#['0-0.75kb' < '0.75-1.25kb' < '1.25-3.5kb' < '>3.5kb' < 'unknown
colors = {'>3.5kb':'red', '0-0.75kb':'green', '0.75-1.25kb':'blue', '1.25-3.5kb':'orange'}
# drop nan values in cpm_merged['length_category']
#cpm_merged = cpm_merged.dropna(subset=['length_category'], how='any')

plt.scatter(cpm_merged['DE_H_H_20_long'], cpm_merged['DE_H_H_17_long'], c =list(cpm_merged['length_category'].map(colors))
             ,alpha=0.5) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5)

# add the pearson correlation coefficient to the plot
corr = cpm_merged['DE_H_H_20_long'].corr(cpm_merged['DE_H_H_17_long'])
corr_spearman = cpm_merged['DE_H_H_20_long'].corr(cpm_merged['DE_H_H_17_long'], method='spearman')
# plot the correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))


plt.xlabel('Kinnex Gene DE_H_H_20 (log2CPM)')
plt.ylabel('Kinnex Gene DE_H_H_17 (log2CPM)')

plt.title('Kinnex gene counts comparison')
plt.xlim(-1, 15)
plt.ylim(-1, 15)
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))

In [ ]:

plt.figure(figsize=(3, 3))
#['0-0.75kb' < '0.75-1.25kb' < '1.25-3.5kb' < '>3.5kb' < 'unknown
colors = {'>3.5kb':'red', '0-0.75kb':'green', '0.75-1.25kb':'blue', '1.25-3.5kb':'orange'}
# drop nan values in cpm_merged['length_category']
#cpm_merged = cpm_merged.dropna(subset=['length_category'], how='any')

plt.scatter(cpm_merged['DE_H_H_20_long'], cpm_merged['DE_H_H_20_short'], c =list(cpm_merged['length_category'].map(colors))
             ,alpha=0.5) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5)

# add the pearson correlation coefficient to the plot
corr = cpm_merged['DE_H_H_20_long'].corr(cpm_merged['DE_H_H_20_short'])
corr_spearman = cpm_merged['DE_H_H_20_long'].corr(cpm_merged['DE_H_H_20_short'], method='spearman')
# plot the correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))

plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=8, label=label) for label, color in colors.items()],
           title='Gene Length Category', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.xlabel('Kinnex Gene DE_H_H_20 (log2CPM)')
plt.ylabel('Illumina Gene DE_H_H_20 (log2CPM)')

plt.title('Kinnex gene counts comparison')
plt.xlim(-1, 15)
plt.ylim(-1, 15)
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 4))

categories = cpm_merged['length_category'].unique()
# drop Na values
categories = categories.dropna()

for idx, category in enumerate(categories):
    ax = axes[idx]
    data = cpm_merged[cpm_merged['length_category'] == category]
    
    ax.scatter(data['DE_C_H_14_long'], data['DE_H_H_18_long'], alpha=0.005)
    
    # Calculate correlations
    corr = data['DE_C_H_14_long'].corr(data['DE_H_H_18_long'])
    corr_spearman = data['DE_C_H_14_long'].corr(data['DE_H_H_18_long'], method='spearman')
    
    # Add text
    ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}\nn = {len(data)}', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', 
            bbox=dict(facecolor='white', alpha=0.5))
    
    ax.set_xlabel('DE_C_H_14 (long) (log2CPM)')
    ax.set_ylabel('DE_H_H_18 (long) (log2CPM)')
    ax.set_title(f'{category}')
    ax.set_xlim(-1, 15)
    ax.set_ylim(-1, 15)
    ax.set_xticks(np.arange(0, 16, 4))
    ax.set_yticks(np.arange(0, 16, 4))

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(3, 3))
plt.scatter(cpm_merged['DE_H_H_19_long'], cpm_merged['DE_H_H_20_long'], alpha=0.5) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5)

# add the pearson correlation coefficient to the plot
corr = cpm_merged['DE_H_H_19_long'].corr(cpm_merged['DE_H_H_20_long'])
# plot the correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson R = {corr:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))

plt.xlabel('Kinnex DE_H_H_19 (log2CPM)')
plt.ylabel('Kinnex DE_H_H_20 (log2CPM)')
plt.xlim(-1, 15)
plt.ylim(-1, 15)
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))


### Correlation of transcript counts between samples

In [ ]:
plt.figure(figsize=(3, 3))
plt.scatter((cpm_merged_transcript['DE_C_H_13_long']), (cpm_merged_transcript['DE_C_H_14_long']), alpha=0.5) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5)

# add the pearson correlation coefficient to the plot
corr = cpm_merged_transcript['DE_C_H_13_long'].corr(cpm_merged_transcript['DE_C_H_14_long'])
# plot the correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))

plt.xlabel('Kinnex transcript DE_C_H_13 (log2CPM)')
plt.ylabel('Kinnex transcript DE_C_H_14 (log2CPM)')
plt.title('Kinnex transcript counts comparison')
plt.xlim(-1, 15)
plt.ylim(-1, 15)
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))


In [ ]:
plt.figure(figsize=(3, 3))
plt.scatter((cpm_merged_transcript['DE_C_H_13_short']), (cpm_merged_transcript['DE_C_H_14_short']), alpha=0.5) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5)

# add the pearson correlation coefficient to the plot
corr = cpm_merged_transcript['DE_C_H_13_short'].corr(cpm_merged_transcript['DE_C_H_14_short'])
# plot the correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))

plt.xlabel('Illumina transcript DE_C_H_13 (log2CPM)')
plt.ylabel('Illumina transcript DE_C_H_14 (log2CPM)')
plt.title('Illumina transcript counts comparison')
plt.xlim(-1, 15)
plt.ylim(-1, 15)
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))

### Correlation of long and short read RNA-seq 

In [ ]:
# sum the cpm counts for all samples (here we only have heat samples)
cpm_merged['sum_long'] = cpm_merged.filter(like='_long').sum(axis=1)
cpm_merged['sum_short'] = cpm_merged.filter(like='_short').sum(axis=1)
cpm_merged_transcript['sum_long'] = cpm_merged_transcript.filter(like='_long').sum(axis=1)
cpm_merged_transcript['sum_short'] = cpm_merged_transcript.filter(like='_short').sum(axis=1)

In [ ]:
# for each sample calcullate the correlation between gene counts

In [ ]:
plt.figure(figsize=(3, 3))

corr = []
  # Exclude the first column which is gene names
markers = ['o', '^', 's', 'p', 'h', '+', 'x', '*']# Different markers for each sample
# plot the differnt samples in the same plot
for sample, marker in zip(sample_names, markers): #sample_names:
    plt.hexbin((cpm_merged[sample + '_long']), (cpm_merged[sample + '_short']),gridsize=(30, 30), cmap='turbo', bins='log', mincnt=1)#, alpha=0.2, label=sample, marker=marker, color = 'blue', s=1) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5
    # get the pearson correlation coefficient for each sample
    corr.append(cpm_merged[sample + '_long'].corr(cpm_merged[sample + '_short']))
    print(sample, corr[-1])

print(corr)
# plot the mean correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {np.mean(corr):.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
plt.xlabel('Kinnex gene (log2CPM)')
plt.ylabel('Illumina gene (log2CPM)')
plt.title('Kinnex vs Illumina gene counts comparison')
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))

### split by legnth category

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

categories = sorted(cpm_merged['length_category'].unique())
markers = ['o', '^', 's', 'p', 'h', '+', 'x', '*']

for idx, category in enumerate(categories):
    ax = axes[idx]
    data = cpm_merged[cpm_merged['length_category'] == category]
    
    corr = []
    
    # Plot all samples on the same hexbin for this category
    for sample, marker in zip(sample_names, markers):
        hb = ax.hexbin( data[sample + '_short'],data[sample + '_long'] ,
                       gridsize=(30, 30), cmap='turbo', bins='log', mincnt=1 )
        
        # Calculate correlation for this sample
        corr.append(data[sample + '_short'].corr(data[sample + '_long']))
        print(f"{category} - {sample}: {corr[-1]:.3f}")
    
    # Add colorbar for each subplot
    plt.colorbar(hb, ax=ax)
    
    # Plot mean correlation
    ax.text(0.05, 0.95, f'Pearson r = {np.mean(corr):.3f}\nn = {len(data)}', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', 
            bbox=dict(facecolor='white', alpha=0.5))
    
    ax.set_ylabel('Kinnex gene (log2CPM)')
    ax.set_xlabel('Illumina gene (log2CPM)')
    ax.set_title(f'{category}')
    ax.set_xticks(np.arange(0, 16, 4))
    ax.set_yticks(np.arange(0, 16, 4))

plt.tight_layout()
plt.show()

print("\nMean correlations by category:")

In [ ]:
plt.figure(figsize=(3, 3))
#colors = {'>3.5kb':'red', '0-1,25kb':'green', '1.25-3.5kb':'blue', 'unknown':'gray'}
# drop nan values in cpm_merged['length_category']
#cpm_merged = cpm_merged.dropna(subset=['length_category'], how='any')

plt.scatter(cpm_merged['DE_H_H_20_short'], cpm_merged['DE_H_H_20_long'], c =list(cpm_merged['length_category'].map(colors))
             ,alpha=0.05) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5)
# legend for the colors
plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', label='>3.5kb', markerfacecolor='red', markersize=10),
                    plt.Line2D([0], [0], marker='o', color='w', label='1.25-3.5kb', markerfacecolor='orange', markersize=10),
                    plt.Line2D([0], [0], marker='o', color='w', label='0.75-1.25kb', markerfacecolor='blue', markersize=10),
                    plt.Line2D([0], [0], marker='o', color='w', label='0-0.75kb', markerfacecolor='green', markersize=10)],
           title='Gene Length Category',bbox_to_anchor=(1.1, 1.05),
            
             fontsize=8)
# add the pearson correlation coefficient to the plot
corr = cpm_merged['DE_H_H_20_long'].corr(cpm_merged['DE_H_H_20_short'])
corr_spearman = cpm_merged['DE_H_H_20_long'].corr(cpm_merged['DE_H_H_20_short'], method='spearman')
# plot the correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.25))


plt.xlabel('Illumina Gene DE_H_H_20 (log2CPM)')
plt.ylabel('Kinnex Gene DE_H_H_20 (log2CPM)')

plt.title('Kinnex gene counts comparison')
plt.xlim(-1, 15)
plt.ylim(-1, 15)
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))

In [ ]:
plt.rcParams['figure.dpi'] = 150
plt.figure(figsize=(3, 3))

corr = []
  # Exclude the first column which is gene names
markers = ['o', '^', 's', 'p', 'h', '+', 'x', '*']# Different markers for each sample
# plot the differnt samples in the same plot
for sample, marker in zip(sample_names, markers): #sample_names:
    plt.hexbin((cpm_merged_transcript[sample + '_long']), (cpm_merged_transcript[sample + '_short']),gridsize=(30, 30), cmap='turbo', bins='log', mincnt=1)#, alpha=0.2, label=sample, marker=marker, color = 'blue', s=1) #cpm_merged['DE_C_H_13_long'], log(cpm_merged['DE_C_H_14_long']), alpha=0.5
    # get the pearson correlation coefficient for each sample
    corr.append(cpm_merged_transcript[sample + '_long'].corr(cpm_merged_transcript[sample + '_short']))

print(corr)
# plot the mean correlation coefficient on the plot
plt.text(0.05, 0.95, f'Pearson r = {np.mean(corr):.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
plt.xlabel('Kinnex gene (log2CPM)')
plt.ylabel('Illumina gene (log2CPM)')
plt.title('Kinnex vs Illumina transcript counts comparison')
plt.xticks(np.arange(0, 16, 4))
plt.yticks(np.arange(0, 16, 4))

In [ ]:
cpm_merged_transcript

### Log2Fold change between heat and control

In [ ]:
# sum the cpm counts for all samples (here we only have heat samples)
cpm_merged['avg_long_heat'] = cpm_merged.filter(regex=r'^DE_H_H.*_long$').sum(axis=1)/4
cpm_merged['avg_long_control'] = cpm_merged.filter(regex=r'^DE_C_H.*_long$').sum(axis=1)/4

cpm_merged['avg_short_heat'] = cpm_merged.filter(regex=r'^DE_H_H.*_short$').sum(axis=1)/4
cpm_merged['avg_short_control'] = cpm_merged.filter(regex=r'^DE_C_H.*_short$').sum(axis=1)/4

In [ ]:
# get the fold change of the transcript
cpm_merged['logFC_long'] = np.log(cpm_merged['avg_long_heat'] / cpm_merged['avg_long_control'])
cpm_merged['logFC_short'] = np.log(cpm_merged['avg_short_heat'] / cpm_merged['avg_short_control'])
cpm_merged['FC_long'] = cpm_merged['avg_long_heat'] / cpm_merged['avg_long_control']
cpm_merged['FC_short'] = cpm_merged['avg_short_heat'] / cpm_merged['avg_short_control']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
categories = sorted(cpm_merged['length_category'].unique())
markers = ['o', '^', 's', 'p', 'h', '+', 'x', '*']
cp_merged_filtered = cpm_merged[(cpm_merged.iloc[:, :16] > 0).all(axis=1)]

FONTSIZE = 15

# First pass: collect all hexbin data to find global vmin/vmax
all_counts = []
for category in categories:
    data = cp_merged_filtered[cp_merged_filtered['length_category'] == category]
    hb_temp = fig.add_axes([0, 0, 0, 0])  # invisible temp axis
    hb = hb_temp.hexbin(data['logFC_short'], data['logFC_long'],
                         gridsize=(30, 30), cmap='turbo', bins='log', mincnt=1)
    all_counts.extend(hb.get_array())
    hb_temp.remove()

global_vmin = min(all_counts)
global_vmax = max(all_counts)

# Second pass: plot with consistent color scale
for idx, category in enumerate(categories):
    ax = axes[idx]
    data = cp_merged_filtered[cp_merged_filtered['length_category'] == category]
    corr = []

    for sample, marker in zip(sample_names, markers):
        hb = ax.hexbin( data['logFC_long'], data['logFC_short'],
                       gridsize=(30, 30), cmap='turbo', bins='log', mincnt=1,
                       vmin=global_vmin, vmax=global_vmax)
        corr.append(data['logFC_short'].corr(data['logFC_long']))
        print(f"{category} - {sample}: {corr[-1]:.3f}")

    cb = plt.colorbar(hb, ax=ax)
    cb.ax.tick_params(labelsize=FONTSIZE - 2)

    ax.text(0.05, 0.95, f'Pearson r = {np.mean(corr):.3f}\nn = {len(data)}',
            transform=ax.transAxes, fontsize=FONTSIZE - 1, verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.5))

    ax.set_xlabel('short-read RNA-seq (log2FC(CPM))', fontsize=FONTSIZE)
    ax.set_ylabel('long-read RNA-seq (log2FC(CPM))', fontsize=FONTSIZE)
    ax.set_title(f'{category}', fontsize=FONTSIZE + 1, fontweight='bold')
    ax.tick_params(axis='both', labelsize=FONTSIZE - 2)
    ax.set_xticks(np.arange(-1, 2, 1))
    ax.set_yticks(np.arange(-1, 2, 1))

plt.tight_layout()
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/Short_vs_long_logFC_corr_gene_length_batch_corrected.svg", dpi=300)
plt.show()

print("\nMean correlations by category:")

# Make histograms of regulation gene sets



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Get regulation patterns
patterns = cpm_merged['RegulationCategory'].unique()
patterns = patterns[~pd.isna(patterns)]  # Remove NaN if any

# Create subplots for each regulation pattern
n_patterns = len(patterns)
fig, axes = plt.subplots(1, n_patterns, figsize=(4*n_patterns, 3))

# Handle case where there's only one pattern
if n_patterns == 1:
    axes = [axes]

for idx, pattern in enumerate(patterns):
    ax = axes[idx]
    
    # Filter data by regulation pattern
    data = cpm_merged[cpm_merged['RegulationCategory'] == pattern]
    
    # Plot histogram with seaborn
    sns.histplot(data=data, x='avgTxLength', bins=100, alpha=0.5, 
                 kde=True, ax=ax, stat='density')
    
    ax.set_xlim(0, 6000)
    ax.set_xlabel('Average Transcript Length (bp)', fontsize=10)
    
    # Only add ylabel to first plot
    if idx == 0:
        ax.set_ylabel('Density', fontsize=10)
    else:
        ax.set_ylabel('')
    
    ax.set_title(f'{pattern}\n(n={len(data)})', fontsize=11)

plt.suptitle('Distribution of Transcript Lengths by Regulation Pattern', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Get regulation patterns
patterns = cpm_merged['RegulationCategory'].unique()
patterns = patterns[~pd.isna(patterns)]  # Remove NaN if any
# less dpi
fig = plt.figure(figsize=(6, 4), dpi=100)
# Create single plot
#plt.figure(figsize=(4, 2))

# Plot histogram for each regulation pattern
    # Filter data by regulation pattern
data = cpm_merged
    
    # Plot histogram with seaborn
sns.kdeplot(data=data, x='avgTxLength',  hue = 'RegulationCategory' , common_norm = False)
plt.xlabel('Average Transcript Length (bp)', fontsize=12)                 
plt.xlim(0, 6000)
#plt.ylabel('Density', fontsize=12)
plt.title('Distribution of Transcript Lengths by Regulation Pattern', fontsize=14)
#plt.legend(fontsize=10, title='Regulation Pattern', title_fontsize=11)
plt.tight_layout()
plt.show()